In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Load general packages
import numpy as np
import matplotlib.pyplot as plt


# Exercise 2: The endogenous grid method (EGM)

Consider Deaton's finite-horizon consumption-saving model:
$$\begin{align*}
V_t(M_t) &= \max_{0\leq C_t\leq M_t}\left\{u(C_t)+\beta\mathbb{E}_t[V_{t+1}(M_{t+1})]\right\},\\
M_{t+1} &= R(M_t-C_t)+Y_{t+1},\\
Y_{t+1} &= \exp(\xi_{t+1}),\qquad
\xi_{t+1}\sim\mathcal{N}(\mu,\sigma_\xi^2),\\
A_t &= M_t-C_t\geq0.
\end{align*}$$

The notebook has two distinct purposes. First, we solve and simulate an economically meaningful stochastic calibration and study its liquidity constraint. Then we deliberately remove uncertainty and impose $\beta R=1$ to construct a **flat-path test** with a known analytical implication. The second calibration is a numerical accuracy test, not the economic baseline.


## 1. Economic baseline: compare the solution methods

Use
$$
\beta=0.96,\qquad R=1.04,\qquad \rho=1,\qquad \sigma_\xi=0.20.
$$
This standard log-utility calibration combines patience with income uncertainty. Solve exactly the same problem using EGM and time iteration.

EGM uses the Euler equation
$$
u'(C_t)=\beta R\,\mathbb{E}_t\left[u'(C_{t+1}^{\star}(M_{t+1}))\right]
$$
to construct the current cash-on-hand grid endogenously. Time iteration instead solves the Euler equation at fixed values of $M_t$.


In [ ]:
import Exercise_1 as ex1
import Exercise_2 as ex2

# Common economic calibration
beta_baseline = 0.96
rho_baseline = 1.0
sigma_baseline = 0.20
R_baseline = 1.04

par_EGM = ex2.setup(beta=beta_baseline, rho=rho_baseline, sigma=sigma_baseline)
par_EGM.R = R_baseline
sol_EGM = ex2.solve_EGM(par_EGM, vector=False)

par_TI = ex1.setup(beta=beta_baseline, rho=rho_baseline, sigma=sigma_baseline)
par_TI.R = R_baseline
sol_TI = ex1.solve_ti(par_TI)


The two methods should produce very similar policies, even though their grids differ. EGM returns an endogenous $M_t$ grid, while time iteration retains the same exogenous grid in every period.

Before plotting, consider:

1. Why does the maximum value of the EGM cash-on-hand grid vary across periods?
2. Where are the largest visible differences between the two numerical solutions?
3. Why is agreement of the policies more important than agreement of their grid points?


In [ ]:
periods = [0, 4, 8]
fig, axes = plt.subplots(1, len(periods), figsize=(15, 4), sharey=True)

for ax, t in zip(axes, periods):
    ax.plot(sol_EGM.M[:, t], sol_EGM.C[:, t], label="EGM", linewidth=2)
    ax.plot(par_TI.grid_M, sol_TI.C[:, t], "--", label="Time iteration", linewidth=2)
    ax.set(title=f"Period {t + 1}", xlabel=r"Cash-on-hand $M_t$", xlim=(0, 5))
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r"Consumption $C_t$")
axes[0].legend()
fig.suptitle("Consumption policies under the economic baseline")
fig.tight_layout()
plt.show()

# Compare policies on a common grid; different native grids should not be compared point by point.
comparison_grid = np.linspace(0.05, 5.0, 500)
print(" period   max |C_EGM-C_TI|")
for t in periods:
    c_egm = np.interp(comparison_grid, sol_EGM.M[:, t], sol_EGM.C[:, t])
    c_ti = np.interp(comparison_grid, par_TI.grid_M, sol_TI.C[:, t])
    print(f" {t + 1:>4d}       {np.max(np.abs(c_egm-c_ti)):.3e}")


## 2. EGM and the liquidity constraint

The liquidity constraint is $A_t=M_t-C_t\geq0$. When it binds,
$$
A_t=0\quad\Longleftrightarrow\quad C_t=M_t,
$$
so every constrained policy segment lies on the same 45-degree line.

EGM begins with a grid for end-of-period assets. The first endogenous point is generated using $A_t\approx0$ and identifies the boundary $(M_t^{cc},C_t^{cc})$ where the constraint ceases to bind. Plot several periods to see how this boundary moves over the horizon. The thick portions of the policies below are the constrained regions; they overlap because all satisfy $C_t=M_t$.


In [ ]:
periods = [0, 4, 7, 8]
colors = plt.cm.viridis(np.linspace(0.10, 0.85, len(periods)))

boundaries = []
for t in periods:
    M_cc = sol_EGM.M[1, t]
    C_cc = sol_EGM.C[1, t]
    boundaries.append((t, M_cc, C_cc, M_cc - C_cc))

fig, (ax, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5.5))

# Full view: the thick constrained segments coincide with the 45-degree line.
m_45 = np.linspace(0, 1.15, 100)
ax.plot(m_45, m_45, "k--", linewidth=1.5, label=r"45-degree line: $C_t=M_t$")

print(" period       M_cc       C_cc      M_cc-C_cc")
for color, (t, M_cc, C_cc, A_cc) in zip(colors, boundaries):
    label = f"Period {t + 1} policy"
    ax.plot(sol_EGM.M[:, t], sol_EGM.C[:, t], color=color,
            linewidth=2, label=label)
    ax.plot([0, M_cc], [0, C_cc], color=color, linewidth=5, alpha=0.45)
    ax.scatter(M_cc, C_cc, color=color, edgecolor="black", s=65, zorder=3)

    # A zoom makes the movement in the nearby boundary points visible.
    ax_zoom.plot(sol_EGM.M[:, t], sol_EGM.C[:, t], color=color, linewidth=2)
    ax_zoom.scatter(M_cc, C_cc, color=color, edgecolor="black", s=75,
                    zorder=3, label=f"Period {t + 1}: $M^{{cc}}={M_cc:.3f}$")
    print(f" {t + 1:>4d}    {M_cc:9.6f}  {C_cc:9.6f}   {A_cc:10.3e}")

ax.set(xlabel=r"Cash-on-hand $M_t$", ylabel=r"Consumption $C_t$",
       xlim=(0, 2.5), ylim=(0, 1.5),
       title="Policies and overlapping constrained regions")
ax.legend(loc="lower right")
ax.grid(alpha=0.2)

M_values = np.array([item[1] for item in boundaries])
zoom_min = M_values.min() - 0.025
zoom_max = M_values.max() + 0.025
ax_zoom.plot([zoom_min, zoom_max], [zoom_min, zoom_max], "k--", linewidth=1.2)
ax_zoom.set(xlabel=r"Cash-on-hand $M_t$", ylabel=r"Consumption $C_t$",
            xlim=(zoom_min, zoom_max), ylim=(zoom_min, zoom_max),
            title=r"Zoom: movement in $(M_t^{cc},C_t^{cc})$")
ax_zoom.legend(loc="upper left")
ax_zoom.grid(alpha=0.2)

fig.suptitle("EGM and the liquidity constraint")
fig.tight_layout()
plt.show()

print(f"\nFirst asset-grid point: {par_EGM.grid_a[0]:.3e}")


## 3. Simulate the stochastic baseline

Simulate $N=10{,}000$ households from $M_1=1.5$ using the baseline EGM policy. With uncertain income, the precautionary-saving motive and the occasionally binding constraint matter. There is therefore no reason for mean consumption to be exactly flat, even though $\beta R$ is close to one.


In [ ]:
sim_baseline = ex2.simulate(par_EGM, sol_EGM)
mean_c_baseline = np.mean(sim_baseline.C, axis=0)
t_grid = np.arange(1, par_EGM.T + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_grid, mean_c_baseline, "-o")
ax.set(xlabel="Period", ylabel="Mean consumption",
       title="Simulated consumption under the stochastic baseline")
ax.grid(alpha=0.2)
plt.show()


## 4. Numerical accuracy: the flat-path test

Now construct a diagnostic case with deterministic income, $\sigma_\xi=0$, and set $R=1/\beta$, so that $\beta R=1$. The Euler equation then implies
$$
u'(C_t)=u'(C_{t+1})\quad\Longrightarrow\quad C_t=C_{t+1}
$$
for an interior solution. This result holds for any strictly concave CRRA utility; we retain $\rho=1$.

This is a comparison with a known analytical solution—not a test designed to make one method fail. We therefore:

1. verify the convergence flags and Euler residuals from time iteration;
2. compare both simulated paths with the exact constant-consumption level; and
3. repeat the calculation as the grids are refined.

The level plot uses a fixed, economically interpretable vertical range. The error plot and refinement table reveal the small numerical differences without letting Matplotlib's axis offset magnify them visually.


In [ ]:
beta_flat = 0.98
rho_flat = 1.0

par_flat_EGM = ex2.setup(beta=beta_flat, rho=rho_flat, sigma=0.0)
sol_flat_EGM = ex2.solve_EGM(par_flat_EGM, vector=True)
sim_flat_EGM = ex2.simulate(par_flat_EGM, sol_flat_EGM)

par_flat_TI = ex1.setup(beta=beta_flat, rho=rho_flat, sigma=0.0)
sol_flat_TI = ex1.solve_ti(par_flat_TI)
sim_flat_TI = ex1.simulate(par_flat_TI, sol_flat_TI)

mean_c_egm = np.mean(sim_flat_EGM.C, axis=0)
mean_c_ti = np.mean(sim_flat_TI.C, axis=0)

# Exact consumption follows from the deterministic intertemporal budget constraint.
discount_factors = par_flat_EGM.R ** (-np.arange(par_flat_EGM.T))
exact_flat_c = (
    par_flat_EGM.M_ini + np.sum(discount_factors[1:])
) / np.sum(discount_factors)

egm_errors = np.abs(mean_c_egm - exact_flat_c)
ti_errors = np.abs(mean_c_ti - exact_flat_c)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(t_grid, mean_c_egm, "-o", label="EGM")
axes[0].plot(t_grid, mean_c_ti, "--s", label="Time iteration")
axes[0].axhline(exact_flat_c, color="black", linewidth=1, label="Exact solution")
axes[0].set(xlabel="Period", ylabel="Mean consumption",
            ylim=(exact_flat_c - 0.01, exact_flat_c + 0.01),
            title="Consumption levels")
axes[0].legend()
axes[0].grid(alpha=0.2)

error_floor = np.finfo(float).eps
axes[1].semilogy(t_grid, np.maximum(egm_errors, error_floor), "-o", label="EGM")
axes[1].semilogy(t_grid, np.maximum(ti_errors, error_floor), "--s", label="Time iteration")
axes[1].set(xlabel="Period", ylabel="Absolute error",
            title="Error relative to the exact flat path")
axes[1].legend()
axes[1].grid(alpha=0.2)

fig.suptitle(r"Flat-path test: $\beta R=1$ and $\sigma_\xi=0$")
fig.tight_layout()
plt.show()

print(f"Exact constant consumption:             {exact_flat_c:.12f}")
print(f"Maximum EGM error:                      {egm_errors.max():.3e}")
print(f"Maximum time-iteration error:           {ti_errors.max():.3e}")
print(f"All time-iteration root solves converged: {sol_flat_TI.root_success.all()}")
print(f"Maximum unconstrained Euler residual:   {sol_flat_TI.root_max_residual.max():.3e}")


### Grid-refinement check

A fair accuracy comparison should not depend on one arbitrary grid. The next cell uses the same number of stored policy points for both methods and reports maximum error relative to the exact solution. Time iteration should become highly accurate as its fixed cash-on-hand grid is refined; EGM obtains high accuracy with fewer points because it places them endogenously where the Euler equation is satisfied.


In [ ]:
def flat_grid_error(num_policy_points):
    """Return flat-path errors using equal numbers of stored policy points."""
    # EGM has one explicit constraint point plus the endogenous points.
    par_egm = ex2.setup(beta=beta_flat, rho=rho_flat, sigma=0.0)
    par_egm.num_a = num_policy_points - 1
    par_egm.grid_a = ex2.nonlinspace(1e-8, par_egm.M, par_egm.num_a, 1.1)
    par_egm.dim = [par_egm.num_a, par_egm.T]
    par_egm.simN = 1
    sol_egm = ex2.solve_EGM(par_egm, vector=True)
    path_egm = ex2.simulate(par_egm, sol_egm).C[0]

    par_ti = ex1.setup(beta=beta_flat, rho=rho_flat, sigma=0.0)
    par_ti.num_M = num_policy_points
    par_ti.grid_M = ex1.nonlinspace(1e-6, par_ti.M, par_ti.num_M, 1.1)
    par_ti.dim = [par_ti.num_M, par_ti.T]
    par_ti.simN = 1
    sol_ti = ex1.solve_ti(par_ti)
    path_ti = ex1.simulate(par_ti, sol_ti).C[0]

    return (
        np.max(np.abs(path_egm - exact_flat_c)),
        np.max(np.abs(path_ti - exact_flat_c)),
        sol_ti.root_success.all(),
        np.max(sol_ti.root_max_residual),
    )


grid_sizes = np.array([50, 100, 175])
egm_grid_errors = []
ti_grid_errors = []

print(" points    EGM max error    TI max error    TI converged    max Euler residual")
for points in grid_sizes:
    egm_error, ti_error, converged, residual = flat_grid_error(points)
    egm_grid_errors.append(egm_error)
    ti_grid_errors.append(ti_error)
    print(f" {points:>5d}      {egm_error:10.3e}     {ti_error:10.3e}"
          f"       {str(converged):>5s}          {residual:10.3e}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(grid_sizes, egm_grid_errors, "-o", label="EGM")
ax.loglog(grid_sizes, ti_grid_errors, "--s", label="Time iteration")
ax.set(xlabel="Number of stored policy points", ylabel="Maximum flat-path error",
       title="Accuracy under grid refinement")
ax.set_xticks(grid_sizes, labels=grid_sizes)
ax.legend()
ax.grid(alpha=0.2, which="both")
plt.show()


## 5. Correctness and speed

Finally compare the loop and vectorized EGM implementations using the economic baseline. Check that they agree before interpreting the timing results. How does the speed gain from EGM matter when solving a model repeatedly—for example, inside an estimator?


In [ ]:
sol_EGM_loop = ex2.solve_EGM(par_EGM, vector=False)
sol_EGM_vec = ex2.solve_EGM(par_EGM, vector=True)
max_policy_difference = np.nanmax(np.abs(sol_EGM_loop.C - sol_EGM_vec.C))
print(f"Maximum loop/vector EGM policy difference: {max_policy_difference:.3e}\n")

print("Time iteration:")
%timeit -n 5 -r 3 ex1.solve_ti(par_TI)

print("EGM with asset-grid loop:")
%timeit -n 5 -r 3 ex2.solve_EGM(par_EGM, vector=False)

print("Vectorized EGM:")
%timeit -n 5 -r 3 ex2.solve_EGM(par_EGM, vector=True)
